In [ ]:
import logging
import tqdm
import random
from exp.run import ExperimentRun
from exp.config import TransformerExperiments, CNNExperiments, LargeTransformerExperiments
logging.basicConfig(level=logging.ERROR)

Three variable should be taken care of:
1. `conf_index`: an index for configureation (1 for CNN, 2 for Transformer, 3 for Qwen3 and Pythia)
2. `total_run`: a number of runs
3. `extire_experiments`:
	1. `False`: a specific configuration will be used, decided by variable `conf_index`.
	2. `True`: Configuration will be also randomly selected during the experiments

In [ ]:
conf_index = 0
total_run: int = 10
entire_experiments: bool = False
configures = [
	CNNExperiments(),
	TransformerExperiments(),
	LargeTransformerExperiments()
]
config = configures[conf_index]

In [ ]:
if entire_experiments:
	# Prepare all Executor instance
	instances = []
	for conf in configures:
		conf.debug = False
		conf.repeats = 5
		conf.gpu_id = 1
		exp = ExperimentRun(config=conf)
		instances.append(exp)

	print(f"Preparing Monte Carlo Data...")
	for i in tqdm.tqdm(range(total_run)):
		instance: ExperimentRun = random.choice(instances)
		instance.prepare_monte_carlo_experiments_data(number=1)

	# Executing all experiments
	for instance in instances:
		print(f"Executing {instance._config.run_id}...")
		if len(instances.jobs) == 0:
			continue
		if isinstance(instance._config, LargeTransformerExperiments):
			instance.run_experiments_solving_compatibility_issue()
		else:
			instance.run_experiments()
		instance.to_evaluation_result()
else:
	exp = ExperimentRun(config=config)
	exp.run_monte_carlo_experiments(total_run)
	exp.to_evaluation_result()


